In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')


## Augmentation 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator

# Load image
img = load_img('/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/with_mask/image_0.png', 
               target_size=(224, 224))

img_array = img_to_array(img)
img_array = img_array / 255.0

plt.figure(figsize=(5, 5))
plt.imshow(img_array)
plt.title('Original Image')
plt.axis('off')
plt.show()

print("Image Shape:", img_array.shape)

In [ ]:
import cv2
import matplotlib.pyplot as plt

path = '/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/with_mask/image_103.png'

# Read with matplotlib (automatically in RGB)
img_rgb = plt.imread(path)

# Perform flips
h_flip = cv2.flip(img_rgb, 1)
v_flip = cv2.flip(img_rgb, 0)
b_flip = cv2.flip(img_rgb, -1)

# Display
fig, axes = plt.subplots(1, 4, figsize=(20, 10))

axes[0].imshow(img_rgb)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(h_flip)
axes[1].set_title("Horizontal Flip")
axes[1].axis('off')

axes[2].imshow(v_flip)
axes[2].set_title("Vertical Flip")
axes[2].axis('off')

axes[3].imshow(b_flip)
axes[3].set_title("Both Axes")
axes[3].axis('off')

plt.show()

In [ ]:
img = load_img('/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/with_mask/image_0.png',
               target_size=(224, 224))
img_array = img_to_array(img)
img_array = img_array / 255.0
img_array = img_array.reshape((1,) + img_array.shape)

datagen = ImageDataGenerator(
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True
)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle('All Augmentations Combined', fontsize=16)

axes[0][0].imshow(img_array[0])
axes[0][0].set_title('Original')
axes[0][0].axis('off')

count = 1
for batch in datagen.flow(img_array, batch_size=1):
    row = count // 5
    col = count  % 5
    axes[row][col].imshow(batch[0])
    axes[row][col].set_title(f'Augmented {count}')
    axes[row][col].axis('off')
    count += 1
    if count == 15:
        break

plt.tight_layout()
plt.show()

In [ ]:

# ================================================================
# STEP 1: PATHS AND SETTINGS
# ================================================================

TRAIN_PATH = "/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset"
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 10


In [ ]:

# ================================================================
# STEP 2: DATA AUGMENTATION AND LOADING
# ================================================================

train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    validation_split   = 0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'training'
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'validation'
)

print("Classes Found:", train_generator.class_indices)
print(f"Training Samples:   {train_generator.samples}")
print(f"Validation Samples: {val_generator.samples}")


In [ ]:

# ================================================================
# STEP 3: SHOW 2 SAMPLE IMAGES
# ================================================================

images, labels = next(train_generator)
class_names    = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i in range(2):
    axes[i].imshow(images[i])
    axes[i].set_title(f"Label: {class_names[int(labels[i])]}", fontsize=14)
    axes[i].axis('off')
plt.suptitle("Sample Images from Dataset", fontsize=16)
plt.tight_layout()
plt.show()

# ================================================================
# STEP 4: LOAD VGG16 (Transfer Learning)
# ================================================================

base_model = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

base_model.trainable = False

print(f"VGG16 Loaded  - Total Layers: {len(base_model.layers)}")

# ================================================================
# STEP 5: BUILD FULL MODEL
# ================================================================

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0001),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

model.summary()


In [ ]:

# ================================================================
# STEP 6: TRAIN THE MODEL
# ================================================================

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = val_generator
)


In [ ]:

# ================================================================
# STEP 7: PLOT TRAINING RESULTS
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
axes[0].set_title('Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
axes[1].set_title('Accuracy Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# STEP 8: PLOT PREDICTIONS ON VALIDATION IMAGES
# ================================================================

val_images, val_labels = next(val_generator)

predictions = model.predict(val_images)
pred_labels = (predictions > 0.5).astype(int).flatten()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes      = axes.flatten()

for i in range(12):
    axes[i].imshow(val_images[i])

    true_label = class_names[int(val_labels[i])]
    pred_label = class_names[pred_labels[i]]
    confidence = predictions[i][0] if pred_labels[i] == 1 else 1 - predictions[i][0]

    color = 'green' if true_label == pred_label else 'red'

    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label}\nConf: {confidence*100:.1f}%",
        color    = color,
        fontsize = 10
    )
    axes[i].axis('off')

plt.suptitle("Predictions  (Green = Correct  |  Red = Wrong)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# STEP 9: FINAL ACCURACY
# ================================================================

val_loss, val_acc = model.evaluate(val_generator)
print(f"\nFinal Validation Loss:     {val_loss:.4f}")
print(f"Final Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_single_image(img_path, model, class_names):
    # 1. Load and resize the image
    img = image.load_img(img_path, target_size=(224, 224))
    
    # 2. Convert to array and normalize (1./255)
    img_array = image.img_to_array(img) / 255.0
    
    # 3. Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # 4. Make prediction
    prediction = model.predict(img_batch)
    
    # 5. Interpret the result
    # For binary classification: 0 is usually 'with_mask' and 1 is 'without_mask' 
    # (depending on your folder names alphabetical order)
    prob = prediction[0][0]
    pred_label = 1 if prob > 0.5 else 0
    confidence = prob if pred_label == 1 else 1 - prob
    
    class_label = class_names[pred_label]
    
    print(f"Prediction: {class_label}")
    print(f"Confidence: {confidence * 100:.2f}%")
    
    # Optional: Display the image
    plt.imshow(img)
    plt.title(f"{class_label} ({confidence * 100:.1f}%)")
    plt.axis('off')
    plt.show()
# Example image path: /kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/without_mask/image_1.png
# ==========================================
# TEST IT ON YOUR IMAGE
# ==========================================
# Replace 'your_test_image.jpg' with your actual file path
my_image_path = "/kaggle/input/datasets/huzifaulhaq/without-mask/images.jpg" 
predict_single_image(my_image_path, model, class_names)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_single_image(img_path, model, class_names):
    # 1. Load and resize the image
    img = image.load_img(img_path, target_size=(224, 224))
    
    # 2. Convert to array and normalize (1./255)
    img_array = image.img_to_array(img) / 255.0
    
    # 3. Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # 4. Make prediction
    prediction = model.predict(img_batch)
    
    # 5. Interpret the result
    # For binary classification: 0 is usually 'with_mask' and 1 is 'without_mask' 
    # (depending on your folder names alphabetical order)
    prob = prediction[0][0]
    pred_label = 1 if prob > 0.5 else 0
    confidence = prob if pred_label == 1 else 1 - prob
    
    class_label = class_names[pred_label]
    
    print(f"Prediction: {class_label}")
    print(f"Confidence: {confidence * 100:.2f}%")
    
    # Optional: Display the image
    plt.imshow(img)
    plt.title(f"{class_label} ({confidence * 100:.1f}%)")
    plt.axis('off')
    plt.show()
# ==========================================
# TEST IT ON YOUR IMAGE
# ==========================================
# Replace 'your_test_image.jpg' with your actual file path
my_image_path = "/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/without_mask/image_1.png" 
predict_single_image(my_image_path, model, class_names)

# Birds Species Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# ================================================================
# STEP 1: PATHS AND SETTINGS
# ================================================================

TRAIN_PATH = "/kaggle/input/datasets/akash2907/bird-species-classification/train_data/train_data"
TEST_PATH  = "/kaggle/input/datasets/akash2907/bird-species-classification/test_data/test_data"
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 100


In [ ]:

# ================================================================
# STEP 2: DATA AUGMENTATION AND LOADING (CATEGORICAL)
# ================================================================

# Augmentation for training
train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 30,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = 'nearest'
)

# Only rescaling for testing/validation
test_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical' # Changed to categorical for many species
)

val_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical'
)

# Get dynamic number of classes and class names
NUM_CLASSES = len(train_generator.class_indices)
class_names = {v: k for k, v in train_generator.class_indices.items()}

print(f"\nTotal Bird Species Detected: {NUM_CLASSES}")
print(f"Training Samples: {train_generator.samples}")

# ================================================================
# STEP 3: LOAD VGG16 (Transfer Learning)
# ================================================================

base_model = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

base_model.trainable = False # Freeze VGG16 layers


In [ ]:

# ================================================================
# STEP 4: BUILD MULTI-CLASS MODEL
# ================================================================

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax') # Softmax gives probabilities for all classes
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0001),
    loss      = 'categorical_crossentropy', # Required for multi-class
    metrics   = ['accuracy']
)

model.summary()


In [ ]:

# ================================================================
# STEP 5: TRAIN THE MODEL
# ================================================================

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = val_generator
)


In [ ]:

# ================================================================
# STEP 6: PLOT TRAINING RESULTS
# ================================================================

def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss Plot
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss History')
    axes[0].legend()
    
    # Accuracy Plot
    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title('Accuracy History')
    axes[1].legend()
    
    plt.show()

plot_history(history)


In [ ]:
def visualize_test_predictions(model, val_generator, class_names, num_samples=4):
    # 1. Grab a batch of images and their true labels
    images, labels = next(val_generator)
    
    # 2. Get model predictions for this batch
    preds = model.predict(images)
    
    # 3. Setup the plot
    plt.figure(figsize=(16, 5))
    
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(images[i])
        
        # Get the index of the true label (from one-hot encoding)
        true_idx = np.argmax(labels[i])
        true_label = class_names[true_idx]
        
        # Get the index of the predicted label
        pred_idx = np.argmax(preds[i])
        pred_label = class_names[pred_idx]
        
        # Calculate confidence percentage
        confidence = preds[i][pred_idx] * 100
        
        # Color code: Green if correct, Red if wrong
        color = 'green' if true_idx == pred_idx else 'red'
        
        plt.title(f"True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%", 
                  color=color, fontsize=10)
        plt.axis('off')
        
    plt.suptitle("Model Predictions on Test Data", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run the function
visualize_test_predictions(model, val_generator, class_names)

In [ ]:

# ================================================================
# STEP 7: PREDICTION FUNCTION FOR SINGLE IMAGE
# ================================================================

def predict_bird_species(img_path, model, class_names):
    # Load and preprocess image
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_array, axis=0) # Add batch dimension
    
    # Make prediction
    preds = model.predict(img_batch)
    top_idx = np.argmax(preds[0]) # Index of highest probability
    confidence = preds[0][top_idx]
    
    result = class_names[top_idx]
    
    print(f"\n--- Prediction Result ---")
    print(f"Species: {result}")
    print(f"Confidence: {confidence*100:.2f}%")
    
    plt.imshow(img)
    plt.title(f"Pred: {result} ({confidence*100:.1f}%)")
    plt.axis('off')
    plt.show()

# To use it, simply provide an image path:


In [ ]:
predict_bird_species("/kaggle/input/datasets/akash2907/bird-species-classification/train_data/train_data/blasti/DSC_6382.jpg", model, class_names)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

train_path = '/kaggle/input/datasets/akash2907/bird-species-classification/train_data/train_data'
test_path = '/kaggle/input/datasets/akash2907/bird-species-classification/test_data/test_data'

train_data = keras.preprocessing.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='training',
    seed=42
)

val_data = keras.preprocessing.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset='validation',
    seed=42
)


In [ ]:

class_names = train_data.class_names
print(f"Classes: {class_names}")

base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = keras.Sequential([
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(16, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20
)


In [ ]:

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

test_images = []
for images, labels in train_data.take(1):
    test_images = images[:2].numpy()
    true_labels = labels[:2].numpy()

predictions = model.predict(test_images)

plt.figure(figsize=(10, 5))
for i in range(2):
    plt.subplot(1, 2, i+1)
    plt.imshow(test_images[i].astype('uint8'))
    pred_class = class_names[np.argmax(predictions[i])]
    confidence = np.max(predictions[i]) * 100
    plt.title(f'Predicted: {pred_class}\nConfidence: {confidence:.2f}%')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import os

data_path = '/kaggle/input/datasets/alxmamaev/flowers-recognition/flowers'

img_size = (224, 224)
batch_size = 32

all_data = keras.preprocessing.image_dataset_from_directory(
    data_path,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=42
)

class_names = all_data.class_names
print(f"Classes: {class_names}")

total_batches = tf.data.experimental.cardinality(all_data).numpy()
train_size = int(0.8 * total_batches)
val_size = int(0.1 * total_batches)
test_size = total_batches - train_size - val_size

train_data = all_data.take(train_size)
remaining = all_data.skip(train_size)
val_data = remaining.take(val_size)
test_data = remaining.skip(val_size)

print(f"Train batches: {train_size}")
print(f"Validation batches: {val_size}")
print(f"Test batches: {test_size}")



In [ ]:

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

base_model = keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)


In [ ]:

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop, reduce_lr]
)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

test_loss, test_acc = model.evaluate(test_data)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'] + history_fine.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'] + history_fine.history['val_accuracy'], label='Val Accuracy', linewidth=2)
plt.axvline(x=len(history.history['accuracy']), color='r', linestyle='--', label='Fine-tuning Start')
plt.title('Model Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'] + history_fine.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'] + history_fine.history['val_loss'], label='Val Loss', linewidth=2)
plt.axvline(x=len(history.history['loss']), color='r', linestyle='--', label='Fine-tuning Start')
plt.title('Model Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(15, 10))
for images, labels in test_data.take(1):
    test_images = images[:9].numpy()
    true_labels = labels[:9].numpy()
    break

predictions = model.predict(test_images)

for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(test_images[i].astype('uint8'))
    pred_class = class_names[np.argmax(predictions[i])]
    true_class = class_names[true_labels[i]]
    confidence = np.max(predictions[i]) * 100
    color = 'green' if pred_class == true_class else 'red'
    plt.title(f'{true_class} → {pred_class}\n{confidence:.1f}%', 
              color=color, fontweight='bold')
    plt.axis('off')
plt.suptitle('Test Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()